# Stage 11 — Naked option buying scale-up (218 stocks + 8 indices, 1H)

Same entry signal as the debit spread strategy (`ema_adx_signal.py`, EMA 9/21 + rising ADX, 1H), but `option_buying_trades.py` exits with a trailing stop instead of a fixed 1.5R target - naked buying has no structural cap the way a spread does, so a fixed target would throw away the whole reason to skip the hedge leg.

**4H trend hard filter added after validation** (`apply_4h_trend_filter()`, baked into `option_buying_scale_pipeline.process_instrument()`/`reprocess_instrument()`): only 1H entries that agree with the symbol's own 4H Supertrend direction at that moment survive. Tested on 12,608 real closed trades matched to their 4H trend state: aligned entries won 53.3% vs 47.2% against, mean R 0.166 vs 0.024, median R +0.08 vs -0.46 - the typical against-trend trade was a loser, the typical aligned trade wasn't. This roughly halves signal frequency by construction, so per-instrument trade counts here will be noticeably lower than a pre-filter run.

A sandbox run (cached 1H data, hand-rolled pandas_ta stand-in since the real library isn't installable there) with the filter applied gave 153 qualified instruments (>=15 closed trades, down from 189 pre-filter), 70.6% positive avg R, mean 0.198, median 0.170 - all higher than the pre-filter numbers, at the cost of a thinner per-instrument sample (mean ~19 closed trades vs ~30+ before). That run used whatever history happened to already be cached per symbol (inconsistent windows), not a clean uniform one. **This notebook does the real, clean version**: same `from_date=2023-01-01` window as `14_debit_spread_scale_all_fno.ipynb` (Stage 9), `force_refresh=True` so every symbol gets the same window regardless of what's currently cached.

New: `src/option_buying_scale_pipeline.py` - parallel to `debit_spread_scale_pipeline.py`, not a modification. Same resume/failure-isolation/rate-limit discipline.

In [1]:
import sys
sys.path.append("..")

from datetime import datetime
from src.auth import get_kite
from src.instruments import get_fno_stock_list, build_nse_token_lookup, get_index_token
from src.option_buying_scale_pipeline import run_all_instruments, build_comparison, save_comparison

kite = get_kite()
token_lookup = build_nse_token_lookup(kite)
stock_list = get_fno_stock_list(kite)
len(stock_list)

216

In [2]:
results = run_all_instruments(
    kite, token_lookup, stock_list,
    from_date=datetime(2023, 1, 1),
    to_date=datetime.now(),
    force_refresh=True
)
results

[1/216] 360ONE — processing...
Fetching 2023-01-01 to 2024-01-31...
Fetching 2024-02-01 to 2025-03-02...
Fetching 2025-03-03 to 2026-04-02...
Fetching 2026-04-03 to 2026-09-01...
Saved 6301 candles to /Users/hgmac/Documents/3_trading/Quant/data/raw/360ONE_60minute.csv
Saved 41 trades to /Users/hgmac/Documents/3_trading/Quant/exports/option_buying/360ONE_trades.csv
[1/216] 360ONE — done (trades=41, win_rate=43.9, avg_r=0.1)
[2/216] ABB — processing...
Fetching 2023-01-01 to 2024-01-31...
Fetching 2024-02-01 to 2025-03-02...
Fetching 2025-03-03 to 2026-04-02...
Fetching 2026-04-03 to 2026-09-01...
Saved 6301 candles to /Users/hgmac/Documents/3_trading/Quant/data/raw/ABB_60minute.csv
Saved 33 trades to /Users/hgmac/Documents/3_trading/Quant/exports/option_buying/ABB_trades.csv
[2/216] ABB — done (trades=33, win_rate=48.5, avg_r=-0.07)
[3/216] ABCAPITAL — processing...
Fetching 2023-01-01 to 2024-01-31...
Fetching 2024-02-01 to 2025-03-02...
Fetching 2025-03-03 to 2026-04-02...
Fetching 20

,total_trades,closed_trades,open_trades,win_rate_pct,avg_r_achieved,median_r_achieved,exit_reason_counts,median_candles_held,pct_trades_trailing_activated,symbol,status,error
0,41.0,41.0,0.0,43.9,0.10,-0.81,"{'STOP': 20, 'TRAIL_STOP': 11, 'TIME': 10}",7.0,41.5,360ONE,SUCCESS,
1,33.0,33.0,0.0,48.5,-0.07,-1.00,"{'STOP': 17, 'TRAIL_STOP': 12, 'TIME': 4}",6.0,45.5,ABB,SUCCESS,
2,42.0,42.0,0.0,50.0,0.02,-0.04,"{'STOP': 20, 'TRAIL_STOP': 19, 'TIME': 3}",6.0,47.6,ABCAPITAL,SUCCESS,
3,34.0,34.0,0.0,64.7,0.50,0.46,"{'TRAIL_STOP': 17, 'STOP': 11, 'TIME': 6}",5.0,61.8,ADANIENSOL,SUCCESS,
4,38.0,37.0,1.0,45.9,-0.02,-1.00,"{'STOP': 20, 'TRAIL_STOP': 12, 'TIME': 5}",4.0,45.9,ADANIENT,SUCCESS,
...,...,...,...,...,...,...,...,...,...,...,...,...
211,28.0,28.0,0.0,57.1,0.48,0.17,"{'TRAIL_STOP': 12, 'STOP': 11, 'TIME': 5}",9.0,57.1,VOLTAS,SUCCESS,
212,18.0,18.0,0.0,61.1,0.24,0.21,"{'TRAIL_STOP': 9, 'STOP': 6, 'TIME': 3}",6.0,55.6,WAAREEENER,SUCCESS,
213,41.0,41.0,0.0,56.1,0.14,0.15,"{'TRAIL_STOP': 20, 'STOP': 18, 'TIME': 3}",7.0,53.7,WIPRO,SUCCESS,
214,49.0,49.0,0.0,40.8,0.09,-1.00,"{'STOP': 29, 'TRAIL_STOP': 18, 'TIME': 2}",5.0,40.8,YESBANK,SUCCESS,


## Indices — same 6 as the debit spread run, separate token lookup

In [3]:
from src.option_buying_scale_pipeline import process_instrument
import pandas as pd

indices = [
    {"symbol": "NIFTY", "tradingsymbol": "NIFTY 50", "exchange": "NSE"},
    {"symbol": "BANKNIFTY", "tradingsymbol": "NIFTY BANK", "exchange": "NSE"},
    {"symbol": "SENSEX", "tradingsymbol": "SENSEX", "exchange": "BSE"},
    {"symbol": "NIFTYNXT50", "tradingsymbol": "NIFTY NEXT 50", "exchange": "NSE"},
    {"symbol": "MIDCPNIFTY", "tradingsymbol": "NIFTY MID SELECT", "exchange": "NSE"},
    {"symbol": "FINNIFTY", "tradingsymbol": "NIFTY FIN SERVICE", "exchange": "NSE"},
]

index_results = []
for idx in indices:
    token = get_index_token(kite, idx["tradingsymbol"], idx["exchange"])
    summary = process_instrument(idx["symbol"], token, kite,
                                  from_date=datetime(2023, 1, 1), to_date=datetime.now())
    summary["status"] = "SUCCESS"
    summary["error"] = ""
    index_results.append(summary)
    print(f"{idx['symbol']} — refreshed")

index_results_df = pd.DataFrame(index_results)
index_results_df

Fetching 2023-01-01 to 2024-01-31...
Fetching 2024-02-01 to 2025-03-02...
Fetching 2025-03-03 to 2026-04-02...
Fetching 2026-04-03 to 2026-09-01...
Saved 6322 candles to /Users/hgmac/Documents/3_trading/Quant/data/raw/NIFTY_60minute.csv
Saved 32 trades to /Users/hgmac/Documents/3_trading/Quant/exports/option_buying/NIFTY_trades.csv
NIFTY — refreshed
Fetching 2023-01-01 to 2024-01-31...
Fetching 2024-02-01 to 2025-03-02...
Fetching 2025-03-03 to 2026-04-02...
Fetching 2026-04-03 to 2026-09-01...
Saved 6322 candles to /Users/hgmac/Documents/3_trading/Quant/data/raw/BANKNIFTY_60minute.csv
Saved 34 trades to /Users/hgmac/Documents/3_trading/Quant/exports/option_buying/BANKNIFTY_trades.csv
BANKNIFTY — refreshed
Fetching 2023-01-01 to 2024-01-31...
Fetching 2024-02-01 to 2025-03-02...
Fetching 2025-03-03 to 2026-04-02...
Fetching 2026-04-03 to 2026-09-01...
Saved 6322 candles to /Users/hgmac/Documents/3_trading/Quant/data/raw/SENSEX_60minute.csv
Saved 27 trades to /Users/hgmac/Documents/3_tr

,total_trades,closed_trades,open_trades,win_rate_pct,avg_r_achieved,median_r_achieved,exit_reason_counts,median_candles_held,pct_trades_trailing_activated,symbol,status,error
0,32,32,0,46.9,0.16,-1.00,"{'STOP': 17, 'TRAIL_STOP': 8, 'TIME': 7}",6.5,40.6,NIFTY,SUCCESS,
1,34,34,0,52.9,0.19,0.16,"{'STOP': 15, 'TRAIL_STOP': 14, 'TIME': 5}",7.0,52.9,BANKNIFTY,SUCCESS,
2,27,27,0,55.6,0.20,0.15,"{'TRAIL_STOP': 11, 'STOP': 10, 'TIME': 6}",7.0,48.1,SENSEX,SUCCESS,
3,36,36,0,58.3,0.32,0.12,"{'TRAIL_STOP': 16, 'STOP': 14, 'TIME': 6}",7.0,58.3,NIFTYNXT50,SUCCESS,
4,41,41,0,53.7,0.33,0.12,"{'STOP': 17, 'TRAIL_STOP': 16, 'TIME': 8}",5.0,51.2,MIDCPNIFTY,SUCCESS,
5,36,36,0,41.7,-0.11,-1.00,"{'STOP': 19, 'TRAIL_STOP': 12, 'TIME': 5}",6.0,41.7,FINNIFTY,SUCCESS,


## Aggregate comparison — the actual verdict

In [4]:
all_results = pd.concat([results, index_results_df], ignore_index=True)
comparison = build_comparison(all_results)
save_comparison(comparison)
comparison

Saved option buying comparison of 216 instruments to /Users/hgmac/Documents/3_trading/Quant/exports/OPTION_BUYING_COMPARISON.xlsx


,symbol,total_trades,closed_trades,win_rate_pct,avg_r_achieved,median_r_achieved,median_candles_held,pct_trades_trailing_activated
0,SWIGGY,18.0,18.0,83.3,1.21,0.94,6.0,77.8
1,HINDUNILVR,37.0,37.0,64.9,0.86,0.77,9.0,56.8
2,PFC,27.0,27.0,70.4,0.82,0.65,7.0,70.4
3,SHRIRAMFIN,38.0,38.0,68.4,0.81,0.20,7.0,63.2
4,RVNL,42.0,42.0,73.8,0.79,0.68,7.0,71.4
...,...,...,...,...,...,...,...,...
211,NTPC,31.0,31.0,51.6,-0.23,0.06,6.0,45.2
212,VMM,24.0,24.0,45.8,-0.25,-0.66,6.0,33.3
213,COALINDIA,41.0,41.0,43.9,-0.25,-1.00,6.0,41.5
214,ASTRAL,27.0,27.0,37.0,-0.30,-1.00,6.0,37.0


In [5]:
# Same >= 15 closed trades floor used everywhere else in this project.
enough_trades = comparison[comparison["closed_trades"] >= 15]

print(f"Instruments processed: {len(comparison)}")
print(f"Instruments with >= 15 closed trades: {len(enough_trades)}")
print(f"Fraction with positive avg_r_achieved: "
      f"{(enough_trades['avg_r_achieved'] > 0).mean() * 100:.1f}%")
print()
print("avg_r_achieved distribution:")
print(enough_trades["avg_r_achieved"].describe())
print()
print("win_rate_pct distribution:")
print(enough_trades["win_rate_pct"].describe())
print()
print("pct_trades_trailing_activated distribution:")
print(enough_trades["pct_trades_trailing_activated"].describe())

Instruments processed: 216
Instruments with >= 15 closed trades: 216
Fraction with positive avg_r_achieved: 80.1%

avg_r_achieved distribution:
count    216.000000
mean       0.192639
std        0.241647
min       -0.330000
25%        0.047500
50%        0.160000
75%        0.312500
max        1.210000
Name: avg_r_achieved, dtype: float64

win_rate_pct distribution:
count    216.000000
mean      54.370370
std        8.231468
min       32.500000
25%       48.600000
50%       54.500000
75%       60.100000
max       83.300000
Name: win_rate_pct, dtype: float64

pct_trades_trailing_activated distribution:
count    216.000000
mean      51.011111
std        8.352410
min       27.500000
25%       45.200000
50%       51.200000
75%       56.875000
max       77.800000
Name: pct_trades_trailing_activated, dtype: float64
